In [ ]:
"""
Fixed-Point Amplitude Amplification - Hamming Weight Barrier Test
==================================================================

Tests whether Fixed-Point AA escapes the Hamming weight barrier.

Key difference from Grover: Fixed-Point converges to target and STAYS there
instead of oscillating. This might reduce ⟨w⟩ after convergence.

Reference: Yoder et al. (2014) "Fixed-Point Quantum Search"
"""

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, amplitude_damping_error
import matplotlib.pyplot as plt
import numpy as np

# =============================================================================
# FIXED-POINT AMPLITUDE AMPLIFICATION
# =============================================================================

def fixed_point_operator(qc, target_qubits, target_state, theta):
    """
    Fixed-point Grover operator with variable rotation angle
    
    Instead of fixed π/2 rotation (standard Grover), uses angle θ
    that changes with iteration to prevent overshooting.
    
    Args:
        qc: QuantumCircuit
        target_qubits: List of qubit indices
        target_state: '111', '000', etc.
        theta: Rotation angle for this iteration
    """
    n = len(target_qubits)
    
    # ===== ORACLE (same as standard Grover) =====
    if target_state == '111':
        qc.h(target_qubits[-1])
        if n == 2:
            qc.cx(target_qubits[0], target_qubits[1])
        elif n == 3:
            qc.ccx(target_qubits[0], target_qubits[1], target_qubits[-1])
        qc.h(target_qubits[-1])
    
    elif target_state == '000':
        qc.x(target_qubits)
        qc.h(target_qubits[-1])
        if n == 2:
            qc.cx(target_qubits[0], target_qubits[1])
        elif n == 3:
            qc.ccx(target_qubits[0], target_qubits[1], target_qubits[-1])
        qc.h(target_qubits[-1])
        qc.x(target_qubits)
    
    # ===== MODIFIED DIFFUSER with angle θ =====
    # Standard diffuser does: 2|s⟩⟨s| - I
    # Fixed-point does: cos(θ)·I + i·sin(θ)·(2|s⟩⟨s| - I)
    
    # Simplified implementation: use Ry gates with angle θ
    # This approximates the fixed-point behavior
    
    qc.h(target_qubits)
    qc.x(target_qubits)
    
    # Apply controlled rotation instead of fixed phase flip
    qc.h(target_qubits[-1])
    if n == 2:
        qc.cx(target_qubits[0], target_qubits[1])
    elif n == 3:
        qc.ccx(target_qubits[0], target_qubits[1], target_qubits[-1])
    qc.h(target_qubits[-1])
    
    # Add phase rotation to implement variable angle
    for q in target_qubits:
        qc.rz(theta, q)
    
    qc.x(target_qubits)
    qc.h(target_qubits)

def create_fixed_point_circuit(n_qubits=3, target_state='111', num_iterations=10):
    """
    Create Fixed-Point Amplitude Amplification circuit
    
    Uses decreasing rotation angles to achieve convergence without oscillation
    
    Args:
        n_qubits: Number of qubits
        target_state: Target to search for
        num_iterations: Number of fixed-point iterations
    """
    qc = QuantumCircuit(n_qubits, n_qubits)
    
    # Initialize uniform superposition
    qc.h(range(n_qubits))
    qc.barrier(label='Init')
    
    # Fixed-point iterations with decreasing angle
    for i in range(num_iterations):
        # Angle schedule: decreases over iterations to prevent overshoot
        # θ_i = π/(2·sqrt(N)) · (1 - i/num_iterations)
        N = 2 ** n_qubits
        theta = (np.pi / (2 * np.sqrt(N))) * (1 - i / num_iterations)
        
        fixed_point_operator(qc, list(range(n_qubits)), target_state, theta)
        qc.barrier(label=f'FP_{i+1}')
    
    qc.measure(range(n_qubits), range(n_qubits))
    
    return qc

# =============================================================================
# SIMPLE FIXED-POINT (Easier to Implement Correctly)
# =============================================================================

def create_simple_fixed_point(n_qubits=3, target_state='111', num_iterations=6):
    """
    Simplified Fixed-Point using the key idea: more iterations with smaller steps
    
    Instead of 2 big Grover iterations, do many small rotations.
    This approximates fixed-point behavior: converges without overshooting.
    """
    qc = QuantumCircuit(n_qubits, n_qubits)
    
    # Initialize
    qc.h(range(n_qubits))
    qc.barrier(label='Init')
    
    # Many small Grover-like operations
    for i in range(num_iterations):
        # Oracle
        if target_state == '111':
            qc.h(n_qubits - 1)
            if n_qubits == 3:
                qc.ccx(0, 1, n_qubits - 1)
            qc.h(n_qubits - 1)
        elif target_state == '000':
            qc.x(range(n_qubits))
            qc.h(n_qubits - 1)
            if n_qubits == 3:
                qc.ccx(0, 1, n_qubits - 1)
            qc.h(n_qubits - 1)
            qc.x(range(n_qubits))
        
        # Partial diffuser (smaller rotation)
        # Use Ry rotations to make smaller steps
        angle = np.pi / (4 * num_iterations)  # Smaller angle per iteration
        
        qc.h(range(n_qubits))
        qc.x(range(n_qubits))
        
        # Controlled phase with reduced angle
        for q in range(n_qubits):
            qc.ry(angle, q)
        
        qc.h(n_qubits - 1)
        if n_qubits == 3:
            qc.ccx(0, 1, n_qubits - 1)
        qc.h(n_qubits - 1)
        
        qc.x(range(n_qubits))
        qc.h(range(n_qubits))
        
        qc.barrier(label=f'FP_{i+1}')
    
    qc.measure(range(n_qubits), range(n_qubits))
    
    return qc

# =============================================================================
# COMPARISON: Standard Grover vs Fixed-Point
# =============================================================================

def compare_grover_vs_fixed_point(n_qubits=3, target_state='111', noise_strength=0.1):
    """
    Direct comparison: Standard Grover vs Fixed-Point AA
    
    This tests: Does Fixed-Point escape the ⟨w⟩ ≈ n/2 barrier?
    """
    print("="*80)
    print(f"COMPARING STANDARD GROVER VS FIXED-POINT")
    print(f"(n={n_qubits}, target={target_state})")
    print("="*80)
    
    # Circuit 1: Standard Grover (baseline)
    qc_grover = QuantumCircuit(n_qubits, n_qubits)
    qc_grover.h(range(n_qubits))
    
    for _ in range(2):  # 2 iterations optimal for 3 qubits
        # Oracle
        if target_state == '111':
            qc_grover.h(n_qubits - 1)
            if n_qubits == 3:
                qc_grover.ccx(0, 1, n_qubits - 1)
            qc_grover.h(n_qubits - 1)
        elif target_state == '000':
            qc_grover.x(range(n_qubits))
            qc_grover.h(n_qubits - 1)
            if n_qubits == 3:
                qc_grover.ccx(0, 1, n_qubits - 1)
            qc_grover.h(n_qubits - 1)
            qc_grover.x(range(n_qubits))
        
        # Diffuser
        qc_grover.h(range(n_qubits))
        qc_grover.x(range(n_qubits))
        qc_grover.h(n_qubits - 1)
        if n_qubits == 3:
            qc_grover.ccx(0, 1, n_qubits - 1)
        qc_grover.h(n_qubits - 1)
        qc_grover.x(range(n_qubits))
        qc_grover.h(range(n_qubits))
    
    qc_grover.measure(range(n_qubits), range(n_qubits))
    
    # Circuit 2: Fixed-Point (more iterations, smaller steps)
    qc_fixed = create_simple_fixed_point(n_qubits, target_state, num_iterations=6)
    
    print(f"\nStandard Grover depth: {qc_grover.depth()}")
    print(f"Fixed-Point depth: {qc_fixed.depth()}")
    
    # Create noise model
    noise_model = NoiseModel()
    amp_error = amplitude_damping_error(noise_strength)
    noise_model.add_all_qubit_quantum_error(amp_error, 
                                           ['u1', 'u2', 'u3', 'cx', 'h', 'x', 'ry', 'rz'])
    
    # Test ideal
    print("\n--- IDEAL (NO NOISE) ---")
    sim_ideal = AerSimulator()
    
    result_grover = sim_ideal.run(qc_grover, shots=4096).result()
    counts_grover = result_grover.get_counts()
    p_grover_ideal = counts_grover.get(target_state, 0) / 4096
    
    result_fixed = sim_ideal.run(qc_fixed, shots=4096).result()
    counts_fixed = result_fixed.get_counts()
    p_fixed_ideal = counts_fixed.get(target_state, 0) / 4096
    
    print(f"Standard Grover P(|{target_state}⟩) = {p_grover_ideal:.4f}")
    print(f"Fixed-Point     P(|{target_state}⟩) = {p_fixed_ideal:.4f}")
    
    # Test noisy
    print(f"\n--- WITH AMPLITUDE DAMPING ({noise_strength*100}%) ---")
    sim_noisy = AerSimulator(noise_model=noise_model)
    
    result_grover_noisy = sim_noisy.run(qc_grover, shots=4096).result()
    counts_grover_noisy = result_grover_noisy.get_counts()
    p_grover_noisy = counts_grover_noisy.get(target_state, 0) / 4096
    
    result_fixed_noisy = sim_noisy.run(qc_fixed, shots=4096).result()
    counts_fixed_noisy = result_fixed_noisy.get_counts()
    p_fixed_noisy = counts_fixed_noisy.get(target_state, 0) / 4096
    
    print(f"Standard Grover P(|{target_state}⟩) = {p_grover_noisy:.4f}")
    print(f"Fixed-Point     P(|{target_state}⟩) = {p_fixed_noisy:.4f}")
    
    # Hamming weight tracking
    print("\n--- HAMMING WEIGHT TRACKING ---")
    
    def track_hamming(qc, noise_model):
        qc_no_meas = qc.remove_final_measurements(inplace=False)
        qc_decomposed = transpile(qc_no_meas, basis_gates=['u1', 'u2', 'u3', 'cx'], 
                                  optimization_level=0)
        
        sim = AerSimulator(method='density_matrix', noise_model=noise_model)
        trajectory = []
        
        step = max(1, len(qc_decomposed.data) // 40)
        measurement_points = list(range(0, len(qc_decomposed.data), step))
        measurement_points.append(len(qc_decomposed.data))
        
        print(f"    Tracking at {len(measurement_points)} points...")
        
        for idx in measurement_points:
            qc_partial = QuantumCircuit(n_qubits)
            
            for i in range(min(idx, len(qc_decomposed.data))):
                instr, qargs, cargs = qc_decomposed.data[i]
                qc_partial.append(instr, qargs, cargs)
            
            qc_partial.save_density_matrix()
            
            try:
                tqc = transpile(qc_partial,sim,optimisation_level=0)
                result = sim.run(tqc, shots=1).result()
                dm = result.data()['density_matrix']
                
                expected_w = 0
                for basis_idx in range(2 ** n_qubits):
                    prob = np.real(dm[basis_idx][basis_idx])
                    w = bin(basis_idx).count('1')
                    expected_w += prob * w
                
                trajectory.append((idx, expected_w))
            except Exception as e:
                print(f"      Error at {idx}: {e}")
        
        return trajectory
    
    print("\nStandard Grover:")
    traj_grover = track_hamming(qc_grover, noise_model)
    w_avg_grover = np.mean([w for _, w in traj_grover])
    w_final_grover = traj_grover[-1][1] if traj_grover else 0
    print(f"  Average ⟨w⟩ = {w_avg_grover:.3f}")
    print(f"  Final ⟨w⟩   = {w_final_grover:.3f}")
    
    print("\nFixed-Point:")
    traj_fixed = track_hamming(qc_fixed, noise_model)
    w_avg_fixed = np.mean([w for _, w in traj_fixed])
    w_final_fixed = traj_fixed[-1][1] if traj_fixed else 0
    print(f"  Average ⟨w⟩ = {w_avg_fixed:.3f}")
    print(f"  Final ⟨w⟩   = {w_final_fixed:.3f}")
    
    # Plot comparison
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Success probabilities
    labels = ['Grover\nIdeal', 'Fixed-Point\nIdeal', 'Grover\nNoisy', 'Fixed-Point\nNoisy']
    values = [p_grover_ideal, p_fixed_ideal, p_grover_noisy, p_fixed_noisy]
    colors = ['green', 'blue', 'orange', 'purple']
    
    bars = ax1.bar(labels, values, color=colors, alpha=0.7, edgecolor='black')
    for bar, val in zip(bars, values):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{val:.3f}', ha='center', va='bottom', fontweight='bold')
    
    ax1.set_ylabel('Success Probability', fontsize=12)
    ax1.set_title(f'Success Probability (target={target_state})', fontsize=13, fontweight='bold')
    ax1.set_ylim(0, 1)
    ax1.grid(axis='y', alpha=0.3)
    
    # Hamming trajectories
    if traj_grover and traj_fixed:
        idx_g, w_g = zip(*traj_grover)
        idx_f, w_f = zip(*traj_fixed)
        
        ax2.plot(idx_g, w_g, 'o-', label='Standard Grover', color='orange', linewidth=2, markersize=4)
        ax2.plot(idx_f, w_f, 's-', label='Fixed-Point', color='purple', linewidth=2, markersize=4)
        ax2.axhline(y=n_qubits/2, color='gray', linestyle='--', 
                   label=f'Expected (n/2={n_qubits/2})', linewidth=2)
        
        # Annotate final values
        ax2.text(idx_g[-1], w_g[-1], f'  {w_g[-1]:.2f}', 
                color='orange', fontweight='bold', va='center')
        ax2.text(idx_f[-1], w_f[-1], f'  {w_f[-1]:.2f}', 
                color='purple', fontweight='bold', va='center')
    
    ax2.set_xlabel('Circuit Depth (instruction index)', fontsize=12)
    ax2.set_ylabel('⟨w(t)⟩', fontsize=12)
    ax2.set_title('Hamming Weight Throughout Execution', fontsize=13, fontweight='bold')
    ax2.legend()
    ax2.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'grover_vs_fixed_point_{target_state}.png', dpi=300, bbox_inches='tight')
    print(f"\n✓ Saved: grover_vs_fixed_point_{target_state}.png")
    plt.show()
    
    # VERDICT
    print("\n" + "="*80)
    print("VERDICT:")
    print("="*80)
    
    if abs(w_avg_fixed - w_avg_grover) < 0.3:
        print("⚠️  Fixed-Point maintains similar ⟨w⟩ ≈ n/2 as Grover!")
        print(f"    Grover:      ⟨w⟩_avg = {w_avg_grover:.3f}")
        print(f"    Fixed-Point: ⟨w⟩_avg = {w_avg_fixed:.3f}")
        print("    → The Hamming weight barrier applies to Fixed-Point too")
        print("    → Even convergent algorithms can't escape")
    else:
        print("✓  Fixed-Point shows DIFFERENT Hamming weight behavior!")
        print(f"    Grover:      ⟨w⟩_avg = {w_avg_grover:.3f}")
        print(f"    Fixed-Point: ⟨w⟩_avg = {w_avg_fixed:.3f}")
        print("    → The barrier might NOT be universal!")
    
    # Check if Fixed-Point converges to higher ⟨w⟩ at the end
    if target_state == '111' and w_final_fixed > w_final_grover + 0.3:
        print("\n🔍 INTERESTING: Fixed-Point shows higher final ⟨w⟩!")
        print(f"    This suggests it converges closer to |{target_state}⟩ (w=3)")
        print("    But average ⟨w⟩ during execution is still ~n/2")
    
    print("="*80)
    
    return {
        'grover': {'ideal': p_grover_ideal, 'noisy': p_grover_noisy, 
                  'w_avg': w_avg_grover, 'w_final': w_final_grover},
        'fixed_point': {'ideal': p_fixed_ideal, 'noisy': p_fixed_noisy, 
                       'w_avg': w_avg_fixed, 'w_final': w_final_fixed}
    }

# =============================================================================
# MAIN
# =============================================================================

if __name__ == "__main__":
    print("\n" + "🎯"*40)
    print("FIXED-POINT AMPLITUDE AMPLIFICATION TEST")
    print("Testing: Does Fixed-Point escape the Hamming weight barrier?")
    print("🎯"*40 + "\n")
    
    results_111 = compare_grover_vs_fixed_point(n_qubits=3, target_state='111', 
                                                noise_strength=0.1)
    
    print("\n" + "-"*80 + "\n")
    
    results_000 = compare_grover_vs_fixed_point(n_qubits=3, target_state='000', 
                                                noise_strength=0.1)
    
    print("\n" + "✅"*40)
    print("FIXED-POINT TEST COMPLETE")
    print("✅"*40)